In [ ]:
# Cell 1 — Imports + config
import json
import numpy as np
import pandas as pd
import fsspec
from pathlib import Path

S3_REGION = "ap-northeast-1"
SO = {"client_kwargs": {"region_name": S3_REGION}}

# StageB base
STAGEB_BASE = "s3://tradebot-config-tokyo/data/stageB/dataset=v1"
META_S3 = f"{STAGEB_BASE}/_meta"

In [ ]:
# Cell 2 — Load meta (_meta) [StageB v1: columns split CSV vs Parquet]
import json
import fsspec

def load_json_s3(path: str) -> dict:
    with fsspec.open(path, "r", **SO) as f:
        return json.load(f)

# Core meta
columns_meta       = load_json_s3(f"{META_S3}/columns.json")           # global contract
columns_xgb_meta   = load_json_s3(f"{META_S3}/columns_xgb.json")       # CSV/XGB schema
columns_pq_meta    = load_json_s3(f"{META_S3}/columns_parquet.json")   # parquet debug schema

dtypes_meta        = load_json_s3(f"{META_S3}/dtypes.json")
split_plan         = load_json_s3(f"{META_S3}/split_plan.json")
feature_groups     = load_json_s3(f"{META_S3}/feature_groups.json")
cleaning_rules     = load_json_s3(f"{META_S3}/cleaning_rules.json")
label_stats        = load_json_s3(f"{META_S3}/label_stats.json")
manifest           = load_json_s3(f"{META_S3}/data_manifest.json")

print("Loaded meta from S3:")
print("columns.json keys:", list(columns_meta.keys()))
print("columns_xgb.json keys:", list(columns_xgb_meta.keys()))
print("columns_parquet.json keys:", list(columns_pq_meta.keys()))
print("dtypes:", len(dtypes_meta))
print("split_plan keys:", split_plan.keys())
print("label_stats keys:", label_stats.keys())
print("manifest keys:", manifest.keys())

# Quick schema summary
label_col = columns_xgb_meta.get("label_col", columns_meta.get("label_col_for_csv", "label_A"))
csv_cols  = columns_xgb_meta.get("csv_cols", columns_meta.get("csv_cols", []))
xgb_feats = columns_xgb_meta.get("feature_cols", columns_meta.get("feature_cols_xgb", []))
pq_cols   = columns_pq_meta.get("parquet_cols", columns_meta.get("parquet_cols", []))
pq_feats  = columns_pq_meta.get("feature_cols_all", columns_meta.get("feature_cols_all", []))

print("\n--- Schema summary ---")
print("LABEL_COL:", label_col)
print("CSV cols:", len(csv_cols), "| XGB feats:", len(xgb_feats))
print("PARQUET cols:", len(pq_cols), "| ALL feats:", len(pq_feats))
print("XGB prefixes:", columns_xgb_meta.get("xgb_feature_prefixes", columns_meta.get("xgb_feature_prefixes", None)))
print("CSV example:", csv_cols[:8])
print("Parquet example:", pq_cols[:8])

In [ ]:
# Cell 3 — Decide “feature columns” + label column (StageB)

# --- debug rapide ---
print("columns_meta type:", type(columns_meta))
if isinstance(columns_meta, dict):
    print("columns_meta keys:", list(columns_meta.keys()))

# --- 1) label col ---
# priorité: label_col_for_csv (stageB columns.json), sinon label_col (columns_xgb.json), sinon fallback
LABEL_COL = None
if isinstance(columns_meta, dict):
    for k in ["label_col_for_csv", "label_col", "LABEL_COL", "label", "target", "y_col"]:
        if k in columns_meta and columns_meta[k]:
            LABEL_COL = columns_meta[k]
            break
if LABEL_COL is None:
    LABEL_COL = "label_A"

# --- 2) colonnes "ALL" (utile si on veut filtrer) ---
ALL_COLS = None
if isinstance(columns_meta, dict):
    for k in ["csv_cols", "parquet_cols", "all_cols", "columns", "cols", "all_columns"]:
        if k in columns_meta and isinstance(columns_meta[k], list):
            ALL_COLS = columns_meta[k]
            break

# --- 3) features ---
# priorité: feature_cols_xgb (pour modèle XGB), sinon feature_cols_all, sinon fallback dtypes
FEATURE_COLS = None
if isinstance(columns_meta, dict):
    for k in ["feature_cols_xgb", "feature_cols_all", "feature_cols", "features", "X_cols", "x_cols", "model_features"]:
        if k in columns_meta and isinstance(columns_meta[k], list) and len(columns_meta[k]) > 0:
            FEATURE_COLS = columns_meta[k]
            break

if FEATURE_COLS is None:
    # fallback: déduction depuis dtypes_meta
    non_feat_prefixes = ("id_", "cfg_", "audit_", "label_")
    all_from_dtypes = list(dtypes_meta.keys())
    FEATURE_COLS = [
        c for c in all_from_dtypes
        if c != LABEL_COL and not c.startswith(non_feat_prefixes)
    ]

# --- 4) sécurité: enlever le label si jamais il s'est glissé dedans ---
FEATURE_COLS = [c for c in FEATURE_COLS if c != LABEL_COL]

# --- 5) optionnel: filtrer sur ALL_COLS si défini (évite erreurs si mismatch meta/fichier) ---
if ALL_COLS is not None:
    missing = [c for c in FEATURE_COLS if c not in ALL_COLS]
    if missing:
        print(f"[warn] {len(missing)} features absentes de ALL_COLS (ex: {missing[:5]}) -> drop")
    FEATURE_COLS = [c for c in FEATURE_COLS if c in ALL_COLS]

print("LABEL_COL:", LABEL_COL)
print("n_feat_cols:", len(FEATURE_COLS))
print("example feat:", FEATURE_COLS[:10])

# --- sanity checks StageB ---
assert LABEL_COL in columns_meta["csv_cols"], "Label absent de csv_cols"
assert all(f in columns_meta["csv_cols"] for f in FEATURE_COLS), "Une feature XGB manque dans csv_cols"
assert len(FEATURE_COLS) == len(set(FEATURE_COLS)), "Doublons dans FEATURE_COLS"
assert all(any(f.startswith(p) for p in columns_meta["xgb_feature_prefixes"]) for f in FEATURE_COLS), \
    "Certaines features ne matchent pas xgb_feature_prefixes"

print("[ok] label + features cohérents avec columns_meta")

In [8]:
# Cell 4 — Helper: list StageB train files from split_plan / manifest

ARTIFACT_KEY = "stageB_parquet"   # ou "stageB_csv_gz"
SPLIT = "train"                  # "train" / "val" / "test" / None
MONTHS = None                    # ex: {"2024-01","2024-02"} ou None

rows = manifest.get("files", []) or []
print("manifest rows:", len(rows))
if len(rows) == 0:
    raise ValueError("manifest['files'] est vide -> rien à lire")

print("example row keys:", list(rows[0].keys()))

# --- filtre ---
rows_f = rows
if SPLIT is not None:
    rows_f = [r for r in rows_f if r.get("split") == SPLIT]
if MONTHS is not None:
    rows_f = [r for r in rows_f if r.get("month") in MONTHS]

print("rows after filter:", len(rows_f))
if len(rows_f) == 0:
    raise ValueError(f"Aucun fichier après filtre (SPLIT={SPLIT}, MONTHS={MONTHS})")

print("example filtered:", rows_f[0])

# --- paths ---
paths = [r.get(ARTIFACT_KEY) for r in rows_f if r.get(ARTIFACT_KEY)]
print("n_paths:", len(paths))
if len(paths) == 0:
    raise ValueError(f"Aucun path trouvé pour ARTIFACT_KEY={ARTIFACT_KEY}. "
                     f"Clés dispo ex: {list(rows_f[0].keys())}")

print("paths example:", paths[:3])

# --- sample ---
SAMPLE_N_FILES = min(3, len(paths))
sample_paths = paths[:SAMPLE_N_FILES]
print("sample_paths:", sample_paths)

# --- colonnes à lire ---
READ_COLS = [LABEL_COL] + FEATURE_COLS[:20]  # garde petit pour sanity
# (optionnel) sécurité: pas de doublons
READ_COLS = list(dict.fromkeys(READ_COLS))

dfs = []
for p in sample_paths:
    if ARTIFACT_KEY == "stageB_parquet":
        dfp = pd.read_parquet(
            p, storage_options=SO, columns=READ_COLS, engine="pyarrow"
        )
    else:
        dfp = pd.read_csv(
            p, storage_options=SO, compression="gzip", usecols=READ_COLS
        )
    dfp["_src"] = p
    dfs.append(dfp)

df_sample = pd.concat(dfs, ignore_index=True)
print("df_sample shape:", df_sample.shape)
display(df_sample.head())

# --- checks rapides ---
missing_cols = [c for c in READ_COLS if c not in df_sample.columns]
if missing_cols:
    raise ValueError(f"Colonnes manquantes dans df_sample: {missing_cols}")

print("label value_counts (head):")
print(df_sample[LABEL_COL].value_counts(dropna=False).head())

# dtypes: features doivent être numériques (float/int)
bad = []
for c in READ_COLS:
    if c == LABEL_COL: 
        continue
    if not (pd.api.types.is_integer_dtype(df_sample[c]) or pd.api.types.is_float_dtype(df_sample[c])):
        bad.append((c, str(df_sample[c].dtype)))
print("non-numeric feature dtypes:", bad[:10])

# --- NaN rates (sample) ---
nan_rate = df_sample[READ_COLS].isna().mean().sort_values(ascending=False)
print("Top NaN rates:")
print(nan_rate.head(15))

# Features avec > 0 NaN
has_nan = nan_rate[nan_rate > 0]
print(f"n_features_with_nan: {len(has_nan)} / {len(READ_COLS)-1}")  # -1 label

manifest rows: 110
example row keys: ['month', 'split', 'stageA_part', 'stageB_parquet', 'stageB_csv_gz', 'n_rows']
rows after filter: 90
example filtered: {'month': '2024-01', 'split': 'train', 'stageA_part': 's3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=01/part-00000.parquet', 'stageB_parquet': 's3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/parquet/BTCUSDT_2024-01_part00000.parquet', 'stageB_csv_gz': 's3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/xgb/BTCUSDT_2024-01_part00000.csv.gz', 'n_rows': 20000}
n_paths: 90
paths example: ['s3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/parquet/BTCUSDT_2024-01_part00000.parquet', 's3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/parquet/BTCUSDT_2024-01_part00001.parquet', 's3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/parquet/BTCUSDT_2024-01_part00002.parquet']
sample_paths: ['s3://tradebot-config-tokyo/data/stageB/dataset=v1/split=train/parquet

,label_A,f_b_depth_ask_L15,f_b_depth_bid_L15,f_b_depth_ratio_L15,f_b_imb_L1,f_b_imb_L15,f_b_imb_L5,f_b_mid_move_10s_bps,f_b_mid_move_30s_bps,f_b_quote_updates_10s,...,f_b_slope_bid_L15,f_b_spread_bps,f_b_spread_vol_30s,f_c_absret_15m_bps,f_c_absret_2m_bps,f_c_absret_5m_bps,f_c_body_to_range_1m,f_c_ema_gap_3_9_bps,f_c_range_1m_bps,_src
0,0,2.430000,21.444998,8.825102,0.911981,0.796440,0.904404,NaN,NaN,1,...,0.931733,0.023633,NaN,NaN,NaN,NaN,0.387446,0.000000,10.913755,s3://tradebot-config-tokyo/data/stageB/dataset...
1,0,3.379000,27.396002,8.107724,0.896774,0.780406,0.902987,2.718712,-0.708986,3,...,0.894583,0.023635,0.000004,NaN,NaN,NaN,0.387446,0.000000,10.913755,s3://tradebot-config-tokyo/data/stageB/dataset...
2,0,6.774999,9.161998,1.352325,0.305189,0.149777,0.233762,2.552106,4.940174,4,...,0.958743,0.023623,0.000002,NaN,NaN,NaN,0.868545,1.310495,5.029469,s3://tradebot-config-tokyo/data/stageB/dataset...
3,0,5.268000,22.463999,4.264236,0.853801,0.620078,0.712919,-0.897599,1.936891,4,...,0.888755,0.023618,0.000002,NaN,NaN,NaN,0.868545,1.310495,5.029469,s3://tradebot-config-tokyo/data/stageB/dataset...
4,0,27.690998,4.092000,0.147774,-0.912670,-0.742504,-0.886341,-0.342187,2.408882,3,...,-0.326491,0.023613,0.056041,NaN,6.685266,NaN,0.462264,2.397297,5.004698,s3://tradebot-config-tokyo/data/stageB/dataset...


label value_counts (head):
label_A
0    59779
1      221
Name: count, dtype: int64
non-numeric feature dtypes: []
Top NaN rates:
f_c_absret_15m_bps       0.000500
f_c_absret_5m_bps        0.000167
f_c_absret_2m_bps        0.000067
f_b_mid_move_10s_bps     0.000017
f_b_mid_move_30s_bps     0.000017
f_b_spread_vol_30s       0.000017
label_A                  0.000000
f_b_slope_bid_L15        0.000000
f_c_ema_gap_3_9_bps      0.000000
f_c_body_to_range_1m     0.000000
f_b_spread_bps           0.000000
f_b_quote_updates_30s    0.000000
f_b_slope_ask_L15        0.000000
f_b_depth_ask_L15        0.000000
f_b_quote_updates_10s    0.000000
dtype: float64
n_features_with_nan: 6 / 20


In [10]:
pos_rate_sample = (df_sample[LABEL_COL] == 1).mean()
print("pos_rate_sample:", pos_rate_sample)

if isinstance(label_stats, dict) and "pos_rate" in label_stats:
    print("pos_rate_meta:", label_stats["pos_rate"])

import pyarrow.parquet as pq

p0 = sample_paths[0]
pf = pq.ParquetFile(p0, filesystem=None)  # pyarrow sait lire via fsspec/stockage déjà géré par pandas? sinon on skip

df_one = pd.read_parquet(sample_paths[0], storage_options=SO,
                         columns=[LABEL_COL] + FEATURE_COLS, engine="pyarrow")
print("df_one shape:", df_one.shape)

pos_rate_sample: 0.003683333333333333
pos_rate_meta: 0.0032895100457176222
df_one shape: (20000, 28)


In [14]:
# Cell 5 — NA ratios + inf check (sur sample) [corrigée StageB]
import numpy as np
import pandas as pd

# 1) features attendues = celles de la cell 3 (XGB)
feat_expected = list(FEATURE_COLS)

# 2) présentes dans df_sample (normalement oui, sauf si READ_COLS tronqué)
feat_present  = [c for c in feat_expected if c in df_sample.columns]
feat_missing  = [c for c in feat_expected if c not in df_sample.columns]

print("n_feat_expected:", len(feat_expected))
print("n_feat_present :", len(feat_present))
print("n_feat_missing :", len(feat_missing))
print("missing example:", feat_missing[:10])

# NA ratio seulement sur celles présentes
na_ratio = df_sample[feat_present].isna().mean().sort_values(ascending=False)
print("Top 15 NA ratios (present feats):")
display(na_ratio.head(15))

# inf check (présentes)
arr = df_sample[feat_present].to_numpy(dtype="float64", copy=False)
print("Any +/-inf in present feats?:", bool(np.isinf(arr).any()))

# 3) check contrat "df_sample" (vu qu'on lit seulement READ_COLS)
READ_COLS = [LABEL_COL] + FEATURE_COLS  # toutes les 27
EXPECTED_SAMPLE = set(READ_COLS).union({"_src"})
present = set(df_sample.columns)

missing_vs_sample_contract = sorted(EXPECTED_SAMPLE - present)
extra_vs_sample_contract   = sorted(present - EXPECTED_SAMPLE)

print("missing vs sample_contract:", missing_vs_sample_contract, "total:", len(missing_vs_sample_contract))
print("extra   vs sample_contract:", extra_vs_sample_contract[:20], "total:", len(extra_vs_sample_contract))

# 4) flags (informative)
cols = set(df_sample.columns)
flags = {
    "has_id": any(c.startswith("id_") for c in cols),
    "has_audit": any(c.startswith("audit_") for c in cols),
    "has_cfg": any(c.startswith("cfg_") for c in cols),
    "has_trades": any(c.startswith("f_t_") for c in cols),
    "has_cross": any(c.startswith("f_x_") for c in cols),
}
print("contract flags:", flags)

n_feat_expected: 27
n_feat_present : 20
n_feat_missing : 7
missing example: ['f_c_range_2m_bps', 'f_c_ret_1m_bps', 'f_c_ret_2m_bps', 'f_c_slope_ema9_bps', 'f_c_vol_15m_bps', 'f_c_vol_5m_bps', 'f_c_wick_ratio_1m']
Top 15 NA ratios (present feats):


f_c_absret_15m_bps       0.000500
f_c_absret_5m_bps        0.000167
f_c_absret_2m_bps        0.000067
f_b_mid_move_10s_bps     0.000017
f_b_mid_move_30s_bps     0.000017
f_b_spread_vol_30s       0.000017
f_b_depth_ask_L15        0.000000
f_b_slope_bid_L15        0.000000
f_c_ema_gap_3_9_bps      0.000000
f_c_body_to_range_1m     0.000000
f_b_spread_bps           0.000000
f_b_slope_ask_L15        0.000000
f_b_depth_bid_L15        0.000000
f_b_quote_updates_30s    0.000000
f_b_quote_updates_10s    0.000000
dtype: float64

Any +/-inf in present feats?: False
missing vs sample_contract: ['f_c_range_2m_bps', 'f_c_ret_1m_bps', 'f_c_ret_2m_bps', 'f_c_slope_ema9_bps', 'f_c_vol_15m_bps', 'f_c_vol_5m_bps', 'f_c_wick_ratio_1m'] total: 7
extra   vs sample_contract: [] total: 0
contract flags: {'has_id': False, 'has_audit': False, 'has_cfg': False, 'has_trades': False, 'has_cross': False}


In [16]:
# Quick check val/test (1 cell)
import numpy as np
import pandas as pd

def load_split_sample(split, n_files=2):
    rows = manifest.get("files", [])
    rows_s = [r for r in rows if r.get("split") == split and r.get(ARTIFACT_KEY)]
    if len(rows_s) == 0:
        raise ValueError(f"No rows for split={split} with key={ARTIFACT_KEY}")
    paths = [r[ARTIFACT_KEY] for r in rows_s][:min(n_files, len(rows_s))]
    dfs=[]
    for p in paths:
        if ARTIFACT_KEY == "stageB_parquet":
            d = pd.read_parquet(p, storage_options=SO, columns=[LABEL_COL]+FEATURE_COLS, engine="pyarrow")
        else:
            d = pd.read_csv(p, storage_options=SO, compression="gzip", usecols=[LABEL_COL]+FEATURE_COLS)
        d["_src"]=p
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

def summarize(df, name):
    X = df[FEATURE_COLS]
    y = df[LABEL_COL]
    pos = int((y==1).sum())
    neg = int((y==0).sum())
    pos_rate = pos / (pos + neg)
    max_nan = X.isna().mean().max()
    any_inf = np.isinf(X.to_numpy(dtype="float64", copy=False)).any()
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("pos/neg:", pos, "/", neg, "pos_rate:", pos_rate)
    print("max_nan_rate_feat:", max_nan)
    print("any_inf:", bool(any_inf))

df_val  = load_split_sample("val", n_files=2)
df_test = load_split_sample("test", n_files=2)

summarize(df_val, "VAL")
summarize(df_test, "TEST")    


=== VAL ===
shape: (40000, 29)
pos/neg: 51 / 39949 pos_rate: 0.001275
max_nan_rate_feat: 0.0006
any_inf: False

=== TEST ===
shape: (40000, 29)
pos/neg: 27 / 39973 pos_rate: 0.000675
max_nan_rate_feat: 0.0006
any_inf: False


In [17]:
# Cell T — Sanity temporelle des splits (manifest)

import pandas as pd

rows = manifest.get("files", [])
dfm = pd.DataFrame(rows)

# extraire une date comparable
dfm["month_dt"] = pd.to_datetime(dfm["month"] + "-01")

summary = (
    dfm.groupby("split")["month_dt"]
    .agg(["min", "max", "count"])
    .sort_values("min")
)

print(summary)

# checks explicites
train_max = summary.loc["train", "max"]
val_min   = summary.loc["val", "min"]
val_max   = summary.loc["val", "max"]
test_min  = summary.loc["test", "min"]

print("\nChecks:")
print("train_max < val_min :", train_max < val_min)
print("val_max   < test_min:", val_max < test_min)

             min        max  count
split                             
train 2024-01-01 2025-06-01     90
val   2025-07-01 2025-08-01     10
test  2025-09-01 2025-10-01     10

Checks:
train_max < val_min : True
val_max   < test_min: True


In [18]:
# Cell F — Feature sanity (distributions)

import numpy as np
import pandas as pd

FEATURE_CHECKS = [
    "f_b_spread_bps",
    "f_b_mid_move_10s_bps",
    "f_c_absret_5m_bps",
]

def load_split_df(split, n_files=2):
    rows = manifest["files"]
    rows_s = [r for r in rows if r["split"] == split]
    paths = [r[ARTIFACT_KEY] for r in rows_s][:n_files]
    dfs = []
    for p in paths:
        d = pd.read_parquet(
            p,
            storage_options=SO,
            columns=FEATURE_CHECKS,
            engine="pyarrow",
        )
        d["_split"] = split
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

df_train = load_split_df("train")
df_val   = load_split_df("val")
df_test  = load_split_df("test")

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

In [19]:
# stats comparatives
summary = (
    df_all
    .groupby("_split")[FEATURE_CHECKS]
    .agg(["mean", "std", "min", "max"])
)

display(summary)

f_b_spread_bps                               f_b_mid_move_10s_bps  \
                 mean       std       min       max                 mean   
_split                                                                     
test         0.009387  0.012521  0.008577  0.924738            -0.015281   
train        0.027658  0.114543  0.002164  8.238156            -0.016485   
val          0.010191  0.037035  0.008124  5.427501             0.006546   

                                         f_c_absret_5m_bps                  \
             std         min         max              mean        std  min   
_split                                                                       
test    1.496982  -30.746115   23.025478          5.867172   6.211320  0.0   
train   4.059551 -162.786865  150.061935         12.786717  16.924124  0.0   
val     1.835139  -26.812881  139.480743          6.156351   7.383132  0.0   

                    
               max  
_split              
test     89.618584  
train   588.958130  
val     248.805923

In [20]:
# quantiles (robustes aux outliers)
quantiles = (
    df_all
    .groupby("_split")[FEATURE_CHECKS]
    .quantile([0.01, 0.5, 0.99])
    .unstack(level=-1)
)

display(quantiles)

f_b_spread_bps                     f_b_mid_move_10s_bps       \
                 0.01      0.50      0.99                 0.01 0.50   
_split                                                                
test         0.008614  0.008972  0.009299            -4.393949  0.0   
train        0.021209  0.022734  0.023766           -10.158829  0.0   
val          0.008187  0.009144  0.009492            -4.784671  0.0   

                  f_c_absret_5m_bps                       
             0.99              0.01      0.50       0.99  
_split                                                    
test     4.285898          0.026291  4.078219  29.031431  
train   10.085988          0.150924  8.625033  74.763549  
val      4.799545          0.036783  4.035708  32.378830